# pyblt — viewing rblt bio-logger files in Python (Google Colab)

Opens the `.h5` files produced by the R package [rblt](https://github.com/sg4r/rblt) (Axytrek, CATS, LUL, WACU, LDATA) as synchronised, zoomable plots, with BORIS behaviour labels shaded on top.

Run the cells from top to bottom (**Runtime ▸ Run all** works too).

*Based on rblt by Sébastien Geiger (CNRS IPHC, GPL ≥ 3).*

## 1. Install packages and get pyblt

In [ ]:
!pip install -q h5py plotly-resampler
![ -f pyblt.py ] || wget -q https://raw.githubusercontent.com/jeantetlorene/Vnet_seaturtles_semisupervised/main/notebooks/pyblt.py
import pyblt

## 2. Point to your data
Put your files in Google Drive and mount it (a pop-up asks for permission), or upload them with the 📁 *Files* panel on the left and use paths like `/content/my_logger.h5`.

The behaviour file is optional: a BORIS *aggregated events* CSV (`Behavior`, `Start (s)`, `Stop (s)`, in seconds from the logger start). Leave `BORIS_FILE` empty if you have none.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Data files
H5_FILE    = "/content/drive/MyDrive/data/my_logger.h5"       #@param {type:"string"}
BORIS_FILE = "/content/drive/MyDrive/data/my_behaviours.csv"  #@param {type:"string"}
SEP        = ","                                              #@param [",", ";"] {allow-input: true}

lg = pyblt.load(H5_FILE, BORIS_FILE or None, sep=SEP)
lg.summary()

## 3. Interactive viewer
Set the options in the form (names as listed by `lg.summary()` above, separated by commas; leave empty for all), then run the cell again to redraw.

Drag on a panel to zoom: all panels follow and the data re-samples at full resolution. Double-click to zoom out.

In [ ]:
#@title Plot options
METRICS   = ""  #@param {type:"string"}
BEHAVIOURS = ""  #@param {type:"string"}
START     = ""  #@param {type:"string"}
END       = ""  #@param {type:"string"}

def _names(s):
    return [x.strip() for x in s.split(",") if x.strip()] or None

pyblt.show(lg, metrics=_names(METRICS), behaviors=_names(BEHAVIOURS),
           start=START or None, end=END or None)

## 4. Save a figure as an interactive HTML file

In [ ]:
fig = pyblt.plot(lg, widget=False)   # same options as above
fig.write_html("pyblt_plot.html")

from google.colab import files
files.download("pyblt_plot.html")

## 5. Work with the data
`lg.data` is a pandas DataFrame indexed by UTC time; `lg.behaviors` holds the behaviour bouts.

In [ ]:
lg.data.head()

In [ ]:
per_second = lg.data.resample("1s").mean()

if lg.behaviors is not None:
    first = lg.behaviors.iloc[0]
    bout = lg.data.loc[first["start"]:first["stop"]]
    print(first["Behavior"], bout.shape)